# HoeffdingTree Missing-Value Policy

This notebook shows how `missing_value_policy` changes prediction in `HoeffdingTree`.

Policies:
- `"default"`: use MOA's original prediction logic
- `"random"`: choose one child randomly when a split feature is missing
- `"all"`: follow all available children and combine their votes

We use `ElectricityTiny()` and inject missing values into the prediction instances.
We summarize the comparison with accuracy and running-time tables.

---

*More information about CapyMOA can be found at* https://www.capymoa.org.

**last update on 16/04/2026**

In [6]:
from time import perf_counter

import numpy as np
import pandas as pd

from capymoa.classifier import HoeffdingTree
from capymoa.datasets import ElectricityTiny
from capymoa.instance import Instance


## Helper Functions

In [7]:
def inject_missing_values(x, rng, missing_rate):
    x_missing = x.copy().astype(float)
    missing_mask = rng.random(x_missing.shape[0]) < missing_rate
    x_missing[missing_mask] = np.nan
    return x_missing


def evaluate_policy(policy, missing_rate, max_instances=1000, seed=7):
    stream = ElectricityTiny()
    schema = stream.get_schema()
    learner = HoeffdingTree(
        schema=schema,
        random_seed=seed,
        missing_value_policy=policy,
    )
    rng = np.random.default_rng(seed)

    correct = 0
    total = 0
    start = perf_counter()

    for i, instance in enumerate(stream):
        if i >= max_instances:
            break

        x_missing = inject_missing_values(instance.x, rng, missing_rate)
        pred_instance = Instance.from_array(schema, x_missing)
        prediction = learner.predict(pred_instance)

        if prediction is not None and prediction == instance.y_index:
            correct += 1
        total += 1

        learner.train(instance)

    return {
        "accuracy": correct / total,
        "runtime_seconds": perf_counter() - start,
    }


## Run the Comparison

In [8]:
rows = []

for missing_rate in [0.0, 0.1, 0.3, 0.5, 0.7]:
    for policy in ["default", "random", "all"]:
        metrics = evaluate_policy(policy, missing_rate)
        rows.append(
            {
                "missing_rate": missing_rate,
                "policy": policy,
                "accuracy": metrics["accuracy"],
                "runtime_seconds": metrics["runtime_seconds"],
            }
        )

results = pd.DataFrame(rows)
accuracy_table = results.pivot(index="missing_rate", columns="policy", values="accuracy")
runtime_table = results.pivot(index="missing_rate", columns="policy", values="runtime_seconds")


## Accuracy

In [9]:
accuracy_table.round(3)


policy,all,default,random
missing_rate,,,
0.0,0.823,0.823,0.823
0.1,0.804,0.805,0.798
0.3,0.741,0.740,0.740
0.5,0.685,0.671,0.677
0.7,0.628,0.607,0.602


## Runtime (seconds)

In [10]:
runtime_table.round(3)


policy,all,default,random
missing_rate,,,
0.0,0.038,0.042,0.057
0.1,0.057,0.036,0.083
0.3,0.250,0.037,0.049
0.5,0.064,0.038,0.051
0.7,0.056,0.034,0.050
